In [6]:
# ============================================
# Cell 1: Count existing samples per family
# - بيقرأ كل الملفات القديمة (hashes + folder)
# - يسأل API عن كل hash
# - يحدد كل ملف تابع لأي ransomware family
# - يحفظ النتيجة في family_counts.json
# ============================================

import requests
import time
import json
import os

URL = "https://mb-api.abuse.ch/api/v1/"
API_KEY = "............"
headers = {"Auth-Key": API_KEY}

HASH_FILE = "hashes.txt"
DATASET_FOLDER = r"C:\Users\Analysis_VM\Desktop\Ransomware_Project\dataset\ransomware"

existing_hashes = set()

if os.path.exists(HASH_FILE):
    with open(HASH_FILE, "r") as f:
        for line in f:
            existing_hashes.add(line.strip())

for f in os.listdir(DATASET_FOLDER):
    existing_hashes.add(f.split(".")[0])

print(f"[INFO] Loaded {len(existing_hashes)} hashes")

family_count_existing = {}

for h in existing_hashes:
    try:
        r = requests.post(
            URL,
            headers=headers,
            data={"query": "get_info", "hash": h},
            timeout=20
        )

        data = r.json()

        if data.get("query_status") == "ok":
            sample = data.get("data", [{}])[0]
            family = sample.get("signature", "unknown")

            family_count_existing[family] = family_count_existing.get(family, 0) + 1

        time.sleep(0.3)

    except:
        continue

with open("family_counts.json", "w") as f:
    json.dump(family_count_existing, f)

print("[INFO] Saved family distribution")
print(family_count_existing)

[INFO] Loaded 1013 hashes
[INFO] Saved family distribution
{'VanHelsing': 4, 'RAWorld': 144, 'HellCat': 3, 'Gunra': 6, 'DBatLoader': 1, 'HelloKitty': 40, 'BlackMatter': 24, 'WannaCry': 152, 'Qilin': 55, 'PythonStealer': 3, 'BianLian': 27, 'Cactus': 46, 'Nitrogen': 10, 'Akira': 108, 'JLocker': 3, 'Fog': 17, 'Chaos': 14, 'HiddenTear': 2, 'Lynx': 1, 'Yurei': 6, 'Phorpiex': 5, 'Medusa': 20, 'Neshta': 10, 'Conti': 2, 'Trigona': 105, 'Rhysida': 15, 'Underground': 6, 'BQTLock': 8, 'DragonForce': 18, 'Zatoxp': 2, 'LockBit': 7, 'PureLogsStealer': 3, 'NanoCore': 6, 'Ransomware': 2, 'BlackBasta': 1, 'NightSpire': 4, 'RustyStealer': 11, 'GhostLocker': 1, 'Arcusmedia': 2, 'INC': 14, 'RemcosRAT': 1, 'Global': 1, 'Zombie': 1, 'XoriumStealer': 1, 'njrat': 1, 'INTERLOCK': 6, 'Bert': 5, 'Cloak': 3, 'Gh0stRAT': 2, 'Expiro': 2, 'Morphues': 1, 'Vatican': 2, 'Heodo': 1, 'WhiteLock': 3, 'Nebula': 2, 'GENTLEMEN': 1, 'QuasarRAT': 1, 'Cephalus': 1, 'Skid': 1, 'Makop': 1, 'Mimic': 2, 'XenoRAT': 1, 'RatonRAT': 1,

In [1]:
#Cell A: Setup
#Run every time restarting the kernel to connect to API 
import os
import json
import time
import requests

URL = "https://mb-api.abuse.ch/api/v1/"

API_KEY = "1c728c8e8599d59ac4c7a0f4d3a92e8f1b24fbee6a6a4f6d"

headers = {
    "Auth-Key": API_KEY
}

HASH_FILE = "hashes.txt"

DATASET_FOLDER = r"C:\Users\Analysis_VM\Desktop\Ransomware_Project\dataset\ransomware"

In [2]:
#Cell B: Load existing hashes
existing_hashes = set()

if os.path.exists(HASH_FILE):
    with open(HASH_FILE, "r") as f:
        for line in f:
            existing_hashes.add(line.strip())

for f in os.listdir(DATASET_FOLDER):
    existing_hashes.add(f.split(".")[0])

print(f"[INFO] Loaded hashes: {len(existing_hashes)}")

[INFO] Loaded hashes: 1305


In [3]:
#Cell C: Load existing mapping
if os.path.exists("family_mapping_clean.json"):
    with open("family_mapping_clean.json", "r") as f:
        family_mapping = json.load(f)

    print(f"[INFO] Loaded mapping: {len(family_mapping)}")

else:
    family_mapping = {}
    print("[WARNING] No mapping found")

[INFO] Loaded mapping: 1304


In [7]:
# ============================================
# Cell 2: Smart Family-Based Collection
# ============================================

import requests
import time
from collections import Counter, defaultdict

all_samples = []

# =========================
# Target families + aliases
# =========================
FAMILY_TAGS = {
    "lockbit":     ["lockbit", "lockbit3", "lockbit2"],
    "akira":       ["akira"],
    "blackcat":    ["blackcat", "alphv", "noberus"],
    "medusa":      ["medusa", "medusalocker"],
    "qilin":       ["qilin", "agenda"],
    "hellokitty":  ["hellokitty", "fivehands"],
    "trigona":     ["trigona"],
    "rhysida":     ["rhysida"],
    "cactus":      ["cactus"],
    "bianlian":    ["bianlian"],
    "blackmatter": ["blackmatter", "darkside"],
}

# =========================
# Safe request with retry
# =========================
def safe_request(data):

    for attempt in range(3):

        try:

            r = requests.post(
                URL,
                headers=headers,
                data=data,
                timeout=30
            )

            if r.status_code == 200:
                return r.json().get("data", [])

        except Exception as e:

            print(f"[RETRY {attempt+1}] {e}")
            time.sleep(2)

    print(f"[ERROR] Failed request: {data}")
    return []

# =========================
# Collect by family tags
# NOTE: single loop — no double collection
# _source_family يتسجل مع كل sample
# =========================
for family, tags in FAMILY_TAGS.items():

    for tag in tags:

        r = safe_request({
            "query": "get_taginfo",
            "tag": tag,
            "limit": 200
        })

        # tag الـ source family على كل sample
        for sample in r:
            sample["_source_family"] = family

        print(f"[INFO] {family}/{tag}: {len(r)}")

        all_samples.extend(r)

        time.sleep(2)

print(f"\n[INFO] Raw collected: {len(all_samples)}")

# =========================
# Dedup by sha256
# =========================
unique = {}

for s in all_samples:

    sha256 = s.get("sha256_hash")

    if sha256:
        unique[sha256] = s

samples = list(unique.values())

print(f"[INFO] After dedup: {len(samples)}")

if len(all_samples) > 0:
    ratio = len(samples) / len(all_samples)
    print(f"[DEBUG] Unique ratio: {ratio:.2f}")

# =========================
# Filtering
# NOTE: بما إننا بنجمع بـ family tags مباشرة
#       الـ _source_family هو الـ source of truth
#       مش محتاجين keyword filter على الـ signature
# =========================
bad_keywords = [
    "adware", "spyware", "miner", "grabber",
    "bot", "stealer", "tesla", "trojan",
    "keylogger", "infostealer", "rat",
]

filtered = []

for s in samples:

    sha256     = s.get("sha256_hash")
    file_type  = (s.get("file_type") or "").lower()
    signature  = s.get("signature")
    source_fam = s.get("_source_family")

    # لازم يكون من family tag معروف
    if not source_fam:
        continue

    # لازم signature موجود
    if not signature:
        continue

    NORMALIZE = {
    "ALPHV":       "blackcat",
    "DarkSide":    "blackmatter",
    "Agenda":      "qilin",
    "FiveHands":   "hellokitty",
    "MedusaLocker":"medusa",
}
    
    signature = NORMALIZE.get(signature, signature)
    s["signature"] = signature 
    
    sig = signature.lower()

    if any(k in sig for k in bad_keywords):
        continue

    # PE files فقط
    if file_type and not any(
        x in file_type 
        for x in ["exe", "dll", "windows", "pe32"]
    ):
        continue

    # skip اللي عندك بالفعل
    if sha256 in existing_hashes:
        continue

    filtered.append(s)

already_have = sum(
    1 for s in samples
    if s.get("sha256_hash") in existing_hashes
)
print(f"[DEBUG] Already in dataset: {already_have}")
print(f"[DEBUG] existing_hashes size: {len(existing_hashes)}")
print(f"\n[INFO] Filtered samples: {len(filtered)}")

# =========================
# Semantic diversity by file size
# SIZE_TOLERANCE = 0.20 (20%)
# بيحتفظ بـ samples بـ file sizes متنوعة
# داخل نفس الـ source_family
# =========================
SIZE_TOLERANCE = 0.20

family_groups = defaultdict(list)

for s in filtered:
    # استخدم _source_family مش signature
    # عشان نجمع المتشابهين في نفس الـ group
    fam = s.get("_source_family")
    family_groups[fam].append(s)

'''diverse = []

for family, items in family_groups.items():

    seen_sizes = []

    # رتب من الأصغر للأكبر عشان الـ selection منتظم
    for item in sorted(items, key=lambda x: x.get("file_size", 0)):

        size = item.get("file_size", 0)

        if size == 0:
            # سيب الـ samples اللي مش عندها size info
            diverse.append(item)
            continue

        is_diverse = all(
            abs(size - s) / max(s, 1) > SIZE_TOLERANCE
            for s in seen_sizes
        )

        if is_diverse or not seen_sizes:
            seen_sizes.append(size)
            diverse.append(item)

print(f"[INFO] Diverse samples: {len(diverse)}")'''

diverse = filtered

# =========================
# Distribution report
# =========================
source_families = [s.get("_source_family") for s in diverse]
signatures      = [s.get("signature") for s in diverse]

print("\n[INFO] Distribution by source family:")
for fam, count in Counter(source_families).most_common():
    print(f"  {fam:<15} -> {count}")

print("\n[INFO] Distribution by signature (top 20):")
for sig, count in Counter(signatures).most_common(20):
    print(f"  {sig:<30} -> {count}")

[INFO] lockbit/lockbit: 200
[INFO] lockbit/lockbit3: 9
[INFO] lockbit/lockbit2: 0
[INFO] akira/akira: 174
[INFO] blackcat/blackcat: 87
[INFO] blackcat/alphv: 6
[INFO] blackcat/noberus: 1
[INFO] medusa/medusa: 44
[INFO] medusa/medusalocker: 80
[INFO] qilin/qilin: 67
[INFO] qilin/agenda: 4
[INFO] hellokitty/hellokitty: 54
[INFO] hellokitty/fivehands: 0
[INFO] trigona/trigona: 116
[INFO] rhysida/rhysida: 31
[INFO] cactus/cactus: 56
[INFO] bianlian/bianlian: 40
[INFO] blackmatter/blackmatter: 200
[INFO] blackmatter/darkside: 71

[INFO] Raw collected: 1240
[INFO] After dedup: 1173
[DEBUG] Unique ratio: 0.95
[DEBUG] Already in dataset: 738
[DEBUG] existing_hashes size: 1305

[INFO] Filtered samples: 282

[INFO] Distribution by source family:
  blackmatter     -> 134
  blackcat        -> 73
  medusa          -> 67
  lockbit         -> 8

[INFO] Distribution by signature (top 20):
  BlackMatter                    -> 83
  BlackCat                       -> 70
  medusa                         -> 

In [8]:
# ============================================
# Cell 3: Download Samples
# ============================================

import os
import pyzipper
import io
import json
import requests
import time
from collections import defaultdict

# =========================
# load mappings
# =========================
if os.path.exists("family_mapping.json"):

    try:
        with open("family_mapping.json", "r") as f:
            family_mapping = json.load(f)

    except:
        family_mapping = {}

else:
    family_mapping = {}

# =========================
# round robin grouping
# =========================
groups = defaultdict(list)

for s in diverse:

    family = s.get("signature")

    groups[family].append(s)

# =========================
# round robin selection
# =========================
TARGET_BATCH = min(200, len(diverse))

selected = []

while len(selected) < TARGET_BATCH:

    added = False

    for family in groups:

        if groups[family]:

            sample = groups[family].pop(0)

            selected.append(sample)

            added = True

        if len(selected) >= TARGET_BATCH:
            break

    if not added:
        break

print(f"[INFO] Selected: {len(selected)}")

# =========================
# download
# =========================
session = requests.Session()

downloaded = 0
failed = 0

for s in selected:

    sha256 = s["sha256_hash"]

    family = s.get("signature")

    try:

        r = session.post(
            URL,
            headers=headers,
            data={
                "query": "get_file",
                "sha256_hash": sha256
            },
            timeout=30
        )

        if r.status_code == 200 and b"file_not_found" not in r.content:

            with pyzipper.AESZipFile(io.BytesIO(r.content)) as z:

                z.pwd = b'infected'

                z.extractall(DATASET_FOLDER)

            existing_hashes.add(sha256)

            with open(HASH_FILE, "a") as f:
                f.write(sha256 + "\n")

            family_mapping[sha256] = family

            downloaded += 1

            print(f"[+] {family} | {sha256[:8]}")

        else:

            failed += 1

        time.sleep(0.3)

    except Exception as e:

        print(f"[ERROR] {e}")

        failed += 1

# =========================
# save mapping
# =========================
with open("family_mapping.json", "w") as f:
    json.dump(family_mapping, f)

print("\n========== REPORT ==========")
print(f"[SUCCESS] Downloaded: {downloaded}")
print(f"[FAILED] Failed: {failed}")

[INFO] Selected: 200
[+] LockBit | 2ecf1fe0
[+] BlackMatter | c690148b
[+] BlackCat | df8d0008
[+] blackcat | cefea76d
[+] medusa | 183e9d0d
[+] Neshta | b2493a58
[+] blackmatter | f3f25af5
[+] LockBit | 54b45f35
[+] BlackMatter | d61af007
[+] BlackCat | bc8832f5
[+] blackcat | 7e363b5f
[+] medusa | ea32d0b2
[+] blackmatter | 2de09a81
[+] LockBit | c3230c24
[+] BlackMatter | 8b95627f
[+] BlackCat | 46daafc5
[+] blackcat | 3d7cf20c
[+] medusa | 1bc0575b
[+] blackmatter | 4098b54c
[+] LockBit | f75674d8
[+] BlackMatter | 745a06b9
[+] BlackCat | 62ae5ad2
[+] medusa | dde3c98b
[+] blackmatter | 1d4c0b32
[+] LockBit | 0d38f8bf
[+] BlackMatter | 13782303
[+] BlackCat | 38d5f4f3
[+] medusa | add28507
[+] blackmatter | 6931b124
[+] LockBit | c6cf5fd8
[+] BlackMatter | 2f521817
[+] BlackCat | 1d6d47bf
[+] medusa | f6586d00
[+] blackmatter | 973dfafc
[+] LockBit | 391a97a2
[+] BlackMatter | 4e965dcc
[+] BlackCat | 4729c95c
[+] medusa | 26af2222
[+] blackmatter | bc32a2cc
[+] LockBit | 80e8defa
[

In [9]:
existing_hashes = set()

for f in os.listdir(DATASET_FOLDER):

    hash_val = f.split(".")[0]

    # لازم SHA256 حقيقي
    if len(hash_val) == 64:

        existing_hashes.add(hash_val)

print("[INFO] Hashes loaded:", len(existing_hashes))

[INFO] Hashes loaded: 1504


In [10]:
# ============================================
# Cell D: Update mapping (incremental)
# - يضيف الجديد فقط
# - ويحفظ كله في نفس الملف
# ============================================

import json
import time
import requests
import os

# =========================
# load old mapping
# =========================
if os.path.exists("family_mapping.json"):

    try:
        with open("family_mapping.json", "r") as f:
            family_mapping = json.load(f)

    except:
        print("[WARNING] mapping corrupted, starting fresh")
        family_mapping = {}

else:
    family_mapping = {}

print(f"[INFO] Existing mappings: {len(family_mapping)}")

# =========================
# detect new hashes only
# =========================
new_hashes = [
    h for h in existing_hashes
    if h not in family_mapping
]

print(f"[INFO] New hashes to query: {len(new_hashes)}")

# =========================
# query API only for new
# =========================
for i, h in enumerate(new_hashes):

    try:
        r = requests.post(
            URL,
            headers=headers,
            data={
                "query": "get_info",
                "hash": h
            },
            timeout=20
        )

        data = r.json()

        if data.get("query_status") == "ok":

            sample = data.get("data", [{}])[0]

            family = sample.get("signature")

            if family:
                family_mapping[h] = family

                print(f"[{i+1}/{len(new_hashes)}] {h[:12]} -> {family}")

        time.sleep(0.5)

    except Exception as e:
        print(f"[ERROR] {h[:12]} -> {e}")

# =========================
# save updated mapping
# =========================
with open("family_mapping.json", "w") as f:
    json.dump(family_mapping, f)

print("\n[SUCCESS] Mapping updated")
print(f"[INFO] Total mappings saved: {len(family_mapping)}")

[INFO] Existing mappings: 1504
[INFO] New hashes to query: 0

[SUCCESS] Mapping updated
[INFO] Total mappings saved: 1504
